# 08_03 EES Framework – Stability Score

## Purpose

This notebook calculates the **Stability pillar** of the Explainability-Ethics Scoring (EES) framework.

The notebook is configurable across:

- German Credit
- Pima Diabetes
- COMPAS

Only the `ACTIVE_DATASET` value should be changed.

## What Stability Measures

Stability measures whether model predictions and explanations remain reasonably consistent under small perturbations to the input data.

In this implementation, stability is operationalised using a local Lipschitz-style sensitivity measure:

```text
local_lipschitz_ratio =
absolute change in predicted probability
/
input perturbation magnitude
```

A lower ratio means the model output changes less under small input changes, which indicates greater stability.

## Link to EES

The output of this notebook feeds into:

- `08_06_ees_framework_ees_ahp_score`
- `08_07_results_consolidation`

## Methodological Design

For each model and selected test instance:

1. Start with the original preprocessed test row.
2. Create multiple small numeric perturbations.
3. Score the perturbed rows using the trained estimator-only model.
4. Measure how much the predicted probability changes.
5. Divide the probability change by the perturbation magnitude.
6. Aggregate the local Lipschitz ratios by model.
7. Convert ratios into a 0–1 stability score where higher is better.

This produces a model-level stability score that is comparable across the five candidate model families.

In [1]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# Select only one active dataset.
# ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "preprocessing_folder": "02_data_preprocessing_german_credit",
        "baseline_folder": "03_baseline_modeling_german_credit",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_german_credit",
        "stability_folder": "08_03_ees_framework_stability_score_german_credit"
    },

    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "preprocessing_folder": "02_data_preprocessing_pima_diabetes",
        "baseline_folder": "03_baseline_modeling_pima_diabetes",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_pima_diabetes",
        "stability_folder": "08_03_ees_framework_stability_score_pima_diabetes"
    },

    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "preprocessing_folder": "02_data_preprocessing_compas",
        "baseline_folder": "03_baseline_modeling_compas",
        "ees_input_folder": "08_01_ees_framework_prepare_inputs_compas",
        "stability_folder": "08_03_ees_framework_stability_score_compas"
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])

Active dataset: COMPAS
Dataset name: compas


In [2]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import joblib

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

print("Libraries imported successfully.")
print("Random seed:", RANDOM_STATE)

Libraries imported successfully.
Random seed: 42


In [3]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["stability_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

PREPROCESSING_RESULTS_PATH = RESULTS_ROOT / CONFIG["preprocessing_folder"]
PREPROCESSING_DATASETS_PATH = PREPROCESSING_RESULTS_PATH / "datasets"

BASELINE_RESULTS_PATH = RESULTS_ROOT / CONFIG["baseline_folder"]
BASELINE_MODELS_PATH = BASELINE_RESULTS_PATH / "models"

EES_INPUT_RESULTS_PATH = RESULTS_ROOT / CONFIG["ees_input_folder"]
EES_INPUT_TABLES_PATH = EES_INPUT_RESULTS_PATH / "tables"
EES_INPUT_DATASETS_PATH = EES_INPUT_RESULTS_PATH / "datasets"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("Baseline models path:", BASELINE_MODELS_PATH)
print("Stability output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/compas
Baseline models path: /content/drive/MyDrive/Master_Research/07_Results/compas/03_baseline_modeling_compas/models
Stability output path: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas


In [4]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def load_target_series(path):
    target_df = read_csv_safely(path, "target file")

    if "target" in target_df.columns:
        return target_df["target"].astype(int)

    if CONFIG["target_name"] in target_df.columns:
        return target_df[CONFIG["target_name"]].astype(int)

    if target_df.shape[1] == 1:
        return target_df.iloc[:, 0].astype(int)

    raise ValueError(
        f"Could not identify target column in {path}. "
        f"Available columns: {target_df.columns.tolist()}"
    )


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)

    print(f"Saved: {output_path}")
    return output_path


def get_model_probabilities(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        return 1 / (1 + np.exp(-scores))

    preds = model.predict(X)
    return preds.astype(float)


def minmax_normalise(series, higher_is_better=True):
    series = pd.Series(series).astype(float)

    min_value = series.min()
    max_value = series.max()

    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series([1.0] * len(series), index=series.index)

    if higher_is_better:
        return (series - min_value) / (max_value - min_value)

    return (max_value - series) / (max_value - min_value)


def classify_stability(score):
    if pd.isna(score):
        return "Not available"
    if score >= 0.80:
        return "High stability"
    if score >= 0.60:
        return "Moderate-high stability"
    if score >= 0.40:
        return "Moderate stability"
    if score >= 0.20:
        return "Low-moderate stability"
    return "Low stability"

## 4. Load Preprocessed Modelling Inputs

Stability is evaluated on the preprocessed feature matrix because the estimator-only models from `03` were trained on that representation.

Tracking/index columns are removed to ensure that only model features are perturbed.

In [5]:
# =========================================================
# 4. Load Preprocessed Modelling Inputs
# =========================================================

X_train_preprocessed = read_csv_safely(
    PREPROCESSING_DATASETS_PATH / "X_train_preprocessed.csv",
    "X_train_preprocessed"
)

X_test_preprocessed = read_csv_safely(
    PREPROCESSING_DATASETS_PATH / "X_test_preprocessed.csv",
    "X_test_preprocessed"
)

y_train = load_target_series(
    PREPROCESSING_DATASETS_PATH / "y_train.csv"
)

y_test = load_target_series(
    PREPROCESSING_DATASETS_PATH / "y_test.csv"
)

PREPROCESSED_TRACKING_COLUMNS = [
    "original_index",
    "index",
    "Unnamed: 0"
]

X_train_preprocessed = (
    X_train_preprocessed
    .drop(columns=PREPROCESSED_TRACKING_COLUMNS, errors="ignore")
    .copy()
)

X_test_preprocessed = (
    X_test_preprocessed
    .drop(columns=PREPROCESSED_TRACKING_COLUMNS, errors="ignore")
    .copy()
)

feature_names = X_train_preprocessed.columns.tolist()

print("X_train_preprocessed:", X_train_preprocessed.shape)
print("X_test_preprocessed:", X_test_preprocessed.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Number of features:", len(feature_names))

display(X_test_preprocessed.head())

X_train_preprocessed: (13996, 30)
X_test_preprocessed: (3500, 30)
y_train: (13996,)
y_test: (3500,)
Number of features: 30


,age,juv_fel_count,juv_misd_count,juv_other_count,priors_count,days_b_screening_arrest,c_days_from_compas,sex_Female,sex_Male,race_African-American,race_Asian,race_Caucasian,race_Hispanic,race_Native American,race_Other,c_charge_degree_(CO3),c_charge_degree_(CT),c_charge_degree_(F1),c_charge_degree_(F2),c_charge_degree_(F3),c_charge_degree_(F5),c_charge_degree_(F6),c_charge_degree_(F7),c_charge_degree_(M1),c_charge_degree_(M2),c_charge_degree_(MO3),c_charge_degree_(NI0),c_charge_degree_(TCX),c_charge_degree_(X),c_charge_degree_Missing
0,-0.437500,0.000000,0.000000,1.000000,0.000000,1.000000,985.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.187500,0.000000,0.000000,0.000000,-0.200000,149.000000,2373.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.937500,0.000000,0.000000,0.000000,0.400000,-4.000000,4.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,0.687500,0.000000,0.000000,0.000000,0.800000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2.375000,0.000000,0.000000,0.000000,-0.200000,-1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [6]:
# =========================================================
# 5. Load Estimator-Only Models
# =========================================================

model_files = {
    "logistic_regression": "logistic_regression.pkl",
    "decision_tree": "decision_tree.pkl",
    "random_forest": "random_forest.pkl",
    "xgboost": "xgboost.pkl",
    "mlp": "mlp.pkl",
}

pretty_model_names = {
    "logistic_regression": "Logistic Regression",
    "decision_tree": "Decision Tree",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
    "mlp": "MLP",
}

models = {}
model_load_rows = []

for model_id, filename in model_files.items():
    model_path = BASELINE_MODELS_PATH / filename

    if model_path.exists():
        models[model_id] = joblib.load(model_path)
        found = True
        status = "Loaded"
    else:
        found = False
        status = "Skipped - model file not found"

    model_load_rows.append({
        "model_id": model_id,
        "model_name": pretty_model_names[model_id],
        "model_file_found": found,
        "model_file_path": str(model_path),
        "status": status
    })

model_load_registry_df = pd.DataFrame(model_load_rows)

display(model_load_registry_df)

save_df(model_load_registry_df, "stability_model_load_registry.csv")

if not models:
    raise ValueError("No estimator-only models were loaded.")

,model_id,model_name,model_file_found,model_file_path,status
0,logistic_regression,Logistic Regression,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
1,decision_tree,Decision Tree,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
2,random_forest,Random Forest,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
3,xgboost,XGBoost,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded
4,mlp,MLP,True,/content/drive/MyDrive/Master_Research/07_Resu...,Loaded


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/stability_model_load_registry.csv


## 6. Stability Configuration

The perturbation design is fixed across datasets for comparability.

`N_PERTURBATIONS_PER_INSTANCE` controls how many perturbed neighbours are generated around each test instance.

`PERTURBATION_SCALE` controls the noise magnitude applied to the preprocessed numeric matrix.

In [7]:
# =========================================================
# 6. Stability Configuration
# =========================================================

N_PERTURBATIONS_PER_INSTANCE = 30
PERTURBATION_SCALE = 0.05

# Use all test instances by default.
# For very large future datasets, this can be capped consistently.
MAX_INSTANCES = None

if MAX_INSTANCES is None:
    stability_instance_positions = list(range(len(X_test_preprocessed)))
else:
    stability_instance_positions = sorted(
        rng.choice(
            len(X_test_preprocessed),
            size=min(MAX_INSTANCES, len(X_test_preprocessed)),
            replace=False
        ).tolist()
    )

stability_config_df = pd.DataFrame([
    {
        "parameter": "RANDOM_STATE",
        "value": RANDOM_STATE,
        "description": "Fixed random seed for reproducible perturbation generation."
    },
    {
        "parameter": "N_PERTURBATIONS_PER_INSTANCE",
        "value": N_PERTURBATIONS_PER_INSTANCE,
        "description": "Number of local perturbations generated per test instance."
    },
    {
        "parameter": "PERTURBATION_SCALE",
        "value": PERTURBATION_SCALE,
        "description": "Standard deviation of Gaussian perturbation applied in preprocessed feature space."
    },
    {
        "parameter": "MAX_INSTANCES",
        "value": "All test instances" if MAX_INSTANCES is None else MAX_INSTANCES,
        "description": "Number of test instances included in the stability analysis."
    }
])

display(stability_config_df)

save_df(stability_config_df, "stability_config.csv")

,parameter,value,description
0,RANDOM_STATE,42,Fixed random seed for reproducible perturbatio...
1,N_PERTURBATIONS_PER_INSTANCE,30,Number of local perturbations generated per te...
2,PERTURBATION_SCALE,0.050000,Standard deviation of Gaussian perturbation ap...
3,MAX_INSTANCES,All test instances,Number of test instances included in the stabi...


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/stability_config.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/stability_config.csv')

## 7. Generate Local Perturbation Stability Evidence

For each model and test instance, the notebook calculates:

```text
mean_output_change
median_output_change
max_output_change
mean_local_lipschitz_ratio
median_local_lipschitz_ratio
max_local_lipschitz_ratio
```

Lower local Lipschitz ratios indicate more stable behaviour.

In [8]:
# =========================================================
# 7. Generate Local Perturbation Stability Evidence
# =========================================================

instance_stability_rows = []
perturbation_detail_rows = []
error_rows = []

X_test_values = X_test_preprocessed.values

for model_id, model in models.items():

    model_name = pretty_model_names[model_id]

    print("=" * 100)
    print(f"Calculating stability for: {model_name}")
    print("=" * 100)

    try:
        original_probabilities = get_model_probabilities(
            model,
            X_test_preprocessed
        )

        for test_pos in stability_instance_positions:

            original_row = X_test_values[test_pos]
            original_probability = float(original_probabilities[test_pos])

            output_changes = []
            perturbation_norms = []
            lipschitz_ratios = []

            for perturbation_id in range(N_PERTURBATIONS_PER_INSTANCE):

                noise = rng.normal(
                    loc=0.0,
                    scale=PERTURBATION_SCALE,
                    size=original_row.shape[0]
                )

                perturbed_row = original_row + noise
                perturbation_norm = float(np.linalg.norm(noise))

                perturbed_df = pd.DataFrame(
                    [perturbed_row],
                    columns=feature_names
                )

                perturbed_probability = float(
                    get_model_probabilities(model, perturbed_df)[0]
                )

                output_change = abs(
                    perturbed_probability - original_probability
                )

                local_lipschitz_ratio = (
                    output_change / perturbation_norm
                    if perturbation_norm > 0
                    else np.nan
                )

                output_changes.append(output_change)
                perturbation_norms.append(perturbation_norm)
                lipschitz_ratios.append(local_lipschitz_ratio)

                perturbation_detail_rows.append({
                    "model_id": model_id,
                    "model_name": model_name,
                    "test_instance_position": test_pos,
                    "perturbation_id": perturbation_id,
                    "original_probability": original_probability,
                    "perturbed_probability": perturbed_probability,
                    "output_change": output_change,
                    "perturbation_norm": perturbation_norm,
                    "local_lipschitz_ratio": local_lipschitz_ratio
                })

            instance_stability_rows.append({
                "model_id": model_id,
                "model_name": model_name,
                "test_instance_position": test_pos,
                "n_perturbations": N_PERTURBATIONS_PER_INSTANCE,
                "mean_output_change": float(np.nanmean(output_changes)),
                "median_output_change": float(np.nanmedian(output_changes)),
                "max_output_change": float(np.nanmax(output_changes)),
                "mean_perturbation_norm": float(np.nanmean(perturbation_norms)),
                "median_perturbation_norm": float(np.nanmedian(perturbation_norms)),
                "mean_local_lipschitz_ratio": float(np.nanmean(lipschitz_ratios)),
                "median_local_lipschitz_ratio": float(np.nanmedian(lipschitz_ratios)),
                "max_local_lipschitz_ratio": float(np.nanmax(lipschitz_ratios))
            })

        print(f"Completed stability calculation for {model_name}.")

    except Exception as e:
        error_rows.append({
            "model_id": model_id,
            "model_name": model_name,
            "error_message": str(e)
        })

        print(f"Stability calculation failed for {model_name}: {e}")

instance_stability_summary_df = pd.DataFrame(instance_stability_rows)
perturbation_detail_df = pd.DataFrame(perturbation_detail_rows)
stability_error_log_df = pd.DataFrame(error_rows)

display(instance_stability_summary_df.head())
display(perturbation_detail_df.head())
display(stability_error_log_df)

print("Instance stability shape:", instance_stability_summary_df.shape)
print("Perturbation detail shape:", perturbation_detail_df.shape)

Calculating stability for: Logistic Regression
Completed stability calculation for Logistic Regression.
Calculating stability for: Decision Tree
Completed stability calculation for Decision Tree.
Calculating stability for: Random Forest
Completed stability calculation for Random Forest.
Calculating stability for: XGBoost
Completed stability calculation for XGBoost.
Calculating stability for: MLP
Completed stability calculation for MLP.


,model_id,model_name,test_instance_position,n_perturbations,mean_output_change,median_output_change,max_output_change,mean_perturbation_norm,median_perturbation_norm,mean_local_lipschitz_ratio,median_local_lipschitz_ratio,max_local_lipschitz_ratio
0,logistic_regression,Logistic Regression,0,30,0.035529,0.027358,0.091465,0.266903,0.270671,0.133175,0.108119,0.343507
1,logistic_regression,Logistic Regression,1,30,0.006670,0.005557,0.024821,0.271900,0.272059,0.024698,0.019578,0.098766
2,logistic_regression,Logistic Regression,2,30,0.049233,0.043184,0.098046,0.264312,0.265760,0.187896,0.177705,0.418506
3,logistic_regression,Logistic Regression,3,30,0.048629,0.042825,0.115510,0.282564,0.281527,0.164577,0.154539,0.355135
4,logistic_regression,Logistic Regression,4,30,0.023383,0.019861,0.054086,0.269017,0.269819,0.086189,0.073520,0.238455


,model_id,model_name,test_instance_position,perturbation_id,original_probability,perturbed_probability,output_change,perturbation_norm,local_lipschitz_ratio
0,logistic_regression,Logistic Regression,0,0,0.244733,0.199034,0.045699,0.247751,0.184457
1,logistic_regression,Logistic Regression,0,1,0.244733,0.256591,0.011858,0.252893,0.046889
2,logistic_regression,Logistic Regression,0,2,0.244733,0.272533,0.027799,0.267123,0.104070
3,logistic_regression,Logistic Regression,0,3,0.244733,0.302559,0.057825,0.244771,0.236242
4,logistic_regression,Logistic Regression,0,4,0.244733,0.254936,0.010202,0.276987,0.036834


""


Instance stability shape: (17500, 12)
Perturbation detail shape: (525000, 9)


## 8. Create Model-Level Stability Summary

The model-level score is calculated from the average local Lipschitz ratio.

Because lower Lipschitz ratios are better, min-max normalisation is applied with `higher_is_better=False`.

This converts the raw sensitivity measure into a 0–1 stability score where higher means more stable.

In [9]:
# =========================================================
# 8. Create Model-Level Stability Summary
# =========================================================

if instance_stability_summary_df.empty:
    model_stability_summary_df = pd.DataFrame()
else:
    model_stability_summary_df = (
        instance_stability_summary_df
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(
            n_instances=("test_instance_position", "nunique"),
            mean_output_change=("mean_output_change", "mean"),
            median_output_change=("median_output_change", "median"),
            max_output_change=("max_output_change", "max"),
            mean_perturbation_norm=("mean_perturbation_norm", "mean"),
            median_perturbation_norm=("median_perturbation_norm", "median"),
            mean_local_lipschitz_ratio=("mean_local_lipschitz_ratio", "mean"),
            median_local_lipschitz_ratio=("median_local_lipschitz_ratio", "median"),
            max_local_lipschitz_ratio=("max_local_lipschitz_ratio", "max")
        )
    )

    model_stability_summary_df["stability_score"] = minmax_normalise(
        model_stability_summary_df["mean_local_lipschitz_ratio"],
        higher_is_better=False
    )

    model_stability_summary_df["stability_score_pct"] = (
        model_stability_summary_df["stability_score"] * 100
    )

    model_stability_summary_df["stability_rank"] = (
        model_stability_summary_df["stability_score"]
        .rank(ascending=False, method="dense")
        .astype(int)
    )

    model_stability_summary_df["stability_band"] = (
        model_stability_summary_df["stability_score"]
        .apply(classify_stability)
    )

    model_stability_summary_df = (
        model_stability_summary_df
        .sort_values(["stability_rank", "model_id"])
        .reset_index(drop=True)
    )

display(model_stability_summary_df)

,model_id,model_name,n_instances,mean_output_change,median_output_change,max_output_change,mean_perturbation_norm,median_perturbation_norm,mean_local_lipschitz_ratio,median_local_lipschitz_ratio,max_local_lipschitz_ratio,stability_score,stability_score_pct,stability_rank,stability_band
0,decision_tree,Decision Tree,3500,0.008595,0.000000,0.810897,0.271562,0.270657,0.031236,0.000000,3.610268,1.000000,100.000000,1,High stability
1,logistic_regression,Logistic Regression,3500,0.035847,0.031113,0.219134,0.271717,0.270979,0.132007,0.116251,0.713597,0.661564,66.156446,2,Moderate-high stability
2,mlp,MLP,3500,0.038214,0.032456,0.283023,0.271664,0.270831,0.140674,0.120986,0.934986,0.632455,63.245514,3,Moderate-high stability
3,xgboost,XGBoost,3500,0.038602,0.026659,0.613785,0.271551,0.270658,0.144497,0.098853,2.787682,0.619616,61.961607,4,Moderate-high stability
4,random_forest,Random Forest,3500,0.089450,0.017688,1.000000,0.271612,0.270508,0.328991,0.063693,4.907182,0.000000,0.000000,5,Low stability


## 9. Stability Component Interpretation

In [10]:
# =========================================================
# 9. Stability Component Interpretation
# =========================================================


stability_interpretation_df = pd.DataFrame([
    {
        "metric": "mean_output_change",
        "interpretation": "Average absolute change in predicted probability under perturbation.",
        "higher_is_better": False
    },
    {
        "metric": "mean_local_lipschitz_ratio",
        "interpretation": "Average probability change divided by perturbation magnitude.",
        "higher_is_better": False
    },
    {
        "metric": "stability_score",
        "interpretation": "Min-max normalised stability score where higher values indicate lower sensitivity.",
        "higher_is_better": True
    }
])

display(stability_interpretation_df)

save_df(stability_interpretation_df, "stability_interpretation.csv")

,metric,interpretation,higher_is_better
0,mean_output_change,Average absolute change in predicted probabili...,False
1,mean_local_lipschitz_ratio,Average probability change divided by perturba...,False
2,stability_score,Min-max normalised stability score where highe...,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/stability_interpretation.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/stability_interpretation.csv')

## 10. Save Stability Outputs

In [11]:
# =========================================================
# 10. Save Stability Outputs
# =========================================================

save_df(model_stability_summary_df, "ees_stability_model_scores.csv")
save_df(model_stability_summary_df, "stability_score_summary.csv")
save_df(instance_stability_summary_df, "instance_stability_summary.csv")
save_df(perturbation_detail_df, "stability_perturbation_detail.csv")
save_df(stability_error_log_df, "stability_error_log.csv")

save_df(model_stability_summary_df, "ees_stability_model_scores.csv", folder=DATASETS_PATH)
save_df(model_stability_summary_df, "stability_score_summary.csv", folder=DATASETS_PATH)
save_df(instance_stability_summary_df, "instance_stability_summary.csv", folder=DATASETS_PATH)
save_df(perturbation_detail_df, "stability_perturbation_detail.csv", folder=DATASETS_PATH)
save_df(stability_error_log_df, "stability_error_log.csv", folder=DATASETS_PATH)

save_excel(
    {
        "model_stability": model_stability_summary_df,
        "instance_stability": instance_stability_summary_df,
        "perturbation_detail": perturbation_detail_df,
        "interpretation": stability_interpretation_df,
        "config": stability_config_df,
        "model_load_registry": model_load_registry_df,
        "error_log": stability_error_log_df
    },
    f"ees_stability_score_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "n_perturbations_per_instance": N_PERTURBATIONS_PER_INSTANCE,
    "perturbation_scale": PERTURBATION_SCALE,
    "max_instances": "all" if MAX_INSTANCES is None else MAX_INSTANCES,
    "stability_definition": "Min-max normalised inverse mean local Lipschitz ratio.",
    "standardisation_note": (
        "This stability notebook uses the same configurable perturbation-based local Lipschitz approach "
        "across German Credit, Pima Diabetes and COMPAS."
    )
}

with open(LOGS_PATH / "ees_stability_score_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "ees_stability_score_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All stability outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/ees_stability_model_scores.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/stability_score_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/instance_stability_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/stability_perturbation_detail.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/tables/stability_error_log.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/datasets/ees_stability_model_scores.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/08_03_ees_framework_stability_score_compas/datasets/stability_score_sum

## 11. Interpretation Note

The Stability pillar evaluates whether a model reacts smoothly to small local perturbations.

A model with high stability has relatively small changes in predicted probability when the input is slightly perturbed.

A model with low stability is more sensitive and may produce less robust explanations or decisions.

This score should not be interpreted as predictive accuracy. It is a robustness signal for the EES framework.

In [ ]:
# =========================================================
# 11. Final Completion Message
# =========================================================

print("=" * 100)
print(f"08_03 Stability Score completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("\nKey outputs:")
print("- ees_stability_model_scores.csv")
print("- stability_score_summary.csv")
print("- instance_stability_summary.csv")
print("- stability_perturbation_detail.csv")
print(f"- ees_stability_score_{DATASET_NAME}_outputs.xlsx")